In [1]:
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
#check role


role = sagemaker.get_execution_role()

print(role)

arn:aws:iam::169569810201:role/LabRole


In [3]:
#check your bucket


s3 = boto3.client("s3")

response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-169569810201


In [4]:
 #package up a trained model to deploy it  AWS SageMaker
#because deploying our model to cloud platforms like AWS SageMaker, Google Cloud Vertex AI, or Azure ML, 
#their systems are hardcoded to look for a single compressed archive.

import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "best_model.joblib",
        arcname="best_model.joblib"
    )

In [5]:
%pip install scikit-learn
%pip install joblib

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
#Upload model to S3, because deployment will take the model form s3

s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-169569810201"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model.tar.gz"
)

In [7]:
#check model can be run or not

from inference import model_fn

model = model_fn("")

In [13]:
 import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel


# ---- EDIT THESE ---------------------------------------------------------
BUCKET = "sagemaker-us-east-1-169569810201"
MODEL_S3_KEY = "model.tar.gz"
ENDPOINT_NAME = "UAS-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        # source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying churn endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)


if __name__ == "__main__":
    main()


Role:      arn:aws:iam::169569810201:role/LabRole
Model URI: s3://sagemaker-us-east-1-169569810201/model.tar.gz
Endpoint:  UAS-endpoint

Deploying churn endpoint (5-8 minutes)...
------!

In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?utm_source=chatgpt.com&region=us-east-1#logsV2:log-groups

In [12]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "UAS-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")


Deleting failed endpoint: UAS-endpoint...
No endpoint found to delete: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "UAS-endpoint".
Deleting endpoint configuration: UAS-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete! You can now safely run your main deploy script.
